## #2. Plot stability results

## Purpose

Generate plots to visualize the stability of lineage statistics to simulated topological error and across tree reconstructed with varying sets of integrations

## Setup

In [161]:
import multiprocessing as mp

import cassiopeia as cas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycea as py
import seaborn as sns
import treedata as td
from devmap.config import embryo_palette, set_theme
from devmap.utils import get_paths, load_data, save_plot
from tqdm.notebook import tqdm

set_theme()
base_path, plots_path, results_path = get_paths("stability")

## Load data

In [60]:
tdata = load_data("topology")

Load jacknife results

In [162]:
prog_jackknife = []
linkage_jackknife = []
for embryo in ["e9.5_r1", "e9.5_r2", "e9.5_r3"]:
    for i in range(0,33):
        if not (results_path / embryo / "jackknife" / f"{embryo}_progenitor_stats_{i:02d}.csv").exists():
            print(f"Missing {embryo} iteration {i:02d}")
            continue
        stats = pd.read_csv(results_path / embryo / "jackknife" / f"{embryo}_progenitor_stats_{i:02d}.csv")
        prog_jackknife.append(stats.assign(embryo=embryo.upper().replace("_", "-"), seed=i))
        linkage = pd.read_csv(results_path / embryo / "jackknife" / f"{embryo}_linkage_{i:02d}.csv", index_col = 0)
        linkage_jackknife.append(linkage.assign(embryo=embryo.upper().replace("_", "-"),iteration=i))
prog_jackknife  = pd.concat(prog_jackknife)
linkage_jackknife = pd.concat(linkage_jackknife)
linkage_jackknife["germ_layer"] = linkage_jackknife.index
linkage_jackknife = linkage_jackknife.melt(id_vars = ["germ_layer", "iteration", "embryo"],
                       var_name = "other_germ_layer",
                       value_name = "linkage",
                       value_vars = ["Ectoderm", "Mesoderm", "Endoderm", "Epiblast"])

Load simulation results

In [ ]:
i = 1
prog_sim = []
linkage_sim = []
for i in range(0,10):
    prog_stats = pd.read_csv(results_path / "simulation" / f"simulation_progenitor_stats_t{i:02d}.csv")
    prog_sim.append(prog_stats.assign(seed=i))
    linkage = pd.read_csv(results_path / "simulation" / f"simulation_linkage_t{i:02d}.csv", index_col = 0)
    linkage_sim.append(linkage.assign(seed=i))
prog_sim = pd.concat(prog_sim)
linkage_sim = pd.concat(linkage_sim)
linkage_sim["germ_layer"] = linkage_sim.index
linkage_sim = linkage_sim.melt(id_vars = ["germ_layer", "seed","realized_misplacement"],
                       var_name = "other_germ_layer",
                       value_name = "linkage",
                       value_vars = ["ectoderm", "mesoderm", "endoderm"])

## Jackknife clade statistics

In [168]:
stat_names = {"n_progenitors": "Number of clades", "mean_size": "Mean clade output", "mean_time": "Mean restriction day"}
for stat, name in stat_names.items():
    fig, ax = plt.subplots(figsize=(0.8,2))
    use = ["Endoderm","Mesoderm","Ectoderm"]
    if stat == "mean_time":
        sns.boxplot(data = prog_jackknife.query("fate in @use"), x = "fate", y = stat, hue = "embryo",ax = ax,
                        palette=embryo_palette, saturation = 1, showfliers = False, order = use, legend = False)
    else:
        sns.barplot(data = prog_jackknife.query("fate in @use"), x = "fate", y = stat, hue = "embryo",ax = ax,
                    palette=embryo_palette, saturation = 1, order = use, legend = False,
                    errorbar=("pi", 95),capsize=0.3,err_kws={"linewidth": 0.8,"color":"black"})
    # Rotate x-axis labels
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_ylabel(name)
    ax.set_xlabel("")
    save_plot(plots_path / f"jackknife_{stat}.svg", fig)

## Jackknife ancestral linkage

In [169]:
mat = linkage_jackknife.groupby(["germ_layer", "other_germ_layer"])["linkage"].mean().reset_index().pivot(index = "germ_layer", columns = "other_germ_layer", values = "linkage")
order = ["Epiblast", "Endoderm", "Mesoderm", "Ectoderm"]
fig, ax = plt.subplots(figsize=(1.5,1.5))
sns.heatmap(mat.loc[order, order], square = True, cmap = "RdBu_r", center = 0, vmax = 1.5, vmin = -1.5, cbar = False, ax = ax)
plt.xlabel("");
plt.ylabel("");
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)
save_plot(plots_path / "jackknife_mean_linkage.svg", fig)

In [171]:
fig, ax = plt.subplots(figsize=(1.3,1.5))
sns.barplot(data=linkage_jackknife.query("germ_layer == 'Ectoderm'"), hue = "embryo",
            x = "linkage", y = "other_germ_layer", palette=embryo_palette, saturation = 1,
            ax=ax, linewidth = 1,legend=False,order = order,
            errorbar=("pi", 95),capsize=0.3,err_kws={"linewidth": 0.8,"color":"black"})
ax.set_ylabel("");
ax.axvline(x=0, color='black', linewidth=1);
ax.set_xticks([-.4,0, 0.4, 0.8]);
save_plot(plots_path / "jackknife_ectoderm_linkage.svg", fig)

## Calculate Jackknife RF distances

In [67]:
N_CORES = 20

def compute_rf_task(args):
    rep, i, clone, size, results_path = args

    embryo = f"E9.5-R{rep}"
    rep_key = f"e9.5_r{rep}"

    jackknife_tdata = td.read_h5td(
        results_path / rep_key / "jackknife" / f"{rep_key}_topology_{i:02d}.h5td"
    )

    rf, rfmax = cas.critique.robinson_foulds(
        tdata.obst[clone],
        jackknife_tdata.obst[clone],
    )

    return {
        "embryo": embryo,
        "clone": clone,
        "rf": rf / rfmax if rfmax else 0,
        "index": i,
        "size": size,
    }


tasks = []

for rep in range(1, 4):
    embryo = f"E9.5-R{rep}"
    clone_sizes = tdata.obs.query("embryo == @embryo").clone.value_counts()

    for i in range(0, 33):
        for clone, size in clone_sizes.items():
            tasks.append((rep, i, clone, size, results_path))

rf_values = []

with mp.Pool(processes=N_CORES) as pool:
    for result in tqdm(
        pool.imap_unordered(compute_rf_task, tasks),
        total=len(tasks),
        desc="Computing RF distances",
    ):
        rf_values.append(result)

rf_values = pd.DataFrame(rf_values)
rf_values = (
    rf_values.groupby(["embryo", "index"])
      .apply(lambda g: (g["rf"] * g["size"]).sum() / g["size"].sum())
      .reset_index(name="rf")
)
rf_values.to_csv(results_path / "jackknife_rf_distances.csv", index=False)

## Plot RF distances

In [ ]:
fig, ax = plt.subplots(figsize=(1.2,2))
sns.barplot(data = rf_values, x = "embryo", y = "rf", hue = "embryo",
            palette=embryo_palette, saturation = 1,
            errorbar=("pi", 95),capsize=0.3,err_kws={"linewidth": 0.8,"color":"black"}, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_xlabel("");
ax.set_ylabel("Robinson-Foulds distance\n(leave-one-out vs all integrations)");
save_plot(plots_path / "jackknife_rf.svg", fig)

## Simulated topological error

In [153]:
for stat in ["n_progenitors", "mean_size", "mean_time"]:
    prog_sim[f"{stat}_error"] = ((prog_sim[stat] - prog_sim.groupby(["fate","seed"])[stat].transform("first"))/
        prog_sim.groupby(["fate","seed"])[stat].transform("first")) * 100
prog_sim["misplacement_bin"] = pd.cut(
    prog_sim["realized_misplacement"],
    bins=np.arange(0, .12, 0.01),
    labels=np.arange(0, .11, 0.01),
    right=True,
    include_lowest=True,
).astype(float) * 100
prog_sim.reset_index(inplace=True)

In [165]:
stat_names = {"n_progenitors": "Number of clades", "mean_size": "Mean clade output", "mean_time": "Mean restriction day"}
for stat, name in stat_names.items():
    fig, ax = plt.subplots(figsize=(1.5,1.8))
    sns.lineplot(data = prog_sim.query("misplacement_bin <= 5"), x= "misplacement_bin", y = f"{stat}_error", ax = ax)
    ax.axvline(x=.965, color='black', linestyle='--')
    ax.set_xlabel("Simulated mismatch (%)")
    ax.set_ylabel(f"{name}\nrelative error (%)");
    ax.set_xticks([0,2.5,5])
    save_plot(plots_path / f"simulation_{stat}_error.svg", fig)